In [1]:
!pip install streamlit
!pip install openai
!pip install gTTS
!pip install audio-recorder-streamlit
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [2]:
from openai import OpenAI

client = OpenAI(
    api_key = "d217acfc-3842-4f43-9eba-2067b6fd21a5",
    base_url = "https://api.sambanova.ai/v1"
)

import os
from gtts import gTTS

# Set the OpenAI API key

def speech_to_text(audio_data):
    """Convert speech audio to text using SpeechRecognition."""
    import speech_recognition as sr

    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_data) as source:
        audio = recognizer.record(source)  # Read the entire audio file
        try:
            transcript = recognizer.recognize_google(audio)
            return transcript
        except sr.UnknownValueError:
            return "Sorry, I could not understand the audio."
        except sr.RequestError as e:
            return f"Could not request results from Google Speech Recognition service; {e}"

def text_to_speech(input_text):
    """Convert text to speech using gTTS."""
    tts = gTTS(text=input_text, lang='en')
    audio_file_path = "response.mp3"
    tts.save(audio_file_path)
    return audio_file_path

def get_answer(messages):
    """Get a response from OpenAI's GPT model."""
    try:
        response = client.chat.completions.create(model="Meta-Llama-3.1-8B-Instruct",
        messages=messages)
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [11]:
import os
import streamlit as st
from audio_recorder_streamlit import audio_recorder

# Initialize session state for managing chat messages
def initialize_session_state():
    if "messages" not in st.session_state:
        st.session_state.messages = [{"role": "assistant", "content": "Hi! How may I assist you today?"}]

initialize_session_state()

st.title("अनुग्रह 🤖 (OpenAI Conversational Chatbot)")

# Create a container for the microphone and audio recording
footer_container = st.container()
with footer_container:
    audio_bytes = audio_recorder()

if audio_bytes:
    with st.spinner("Transcribing..."):
        # Write the audio bytes to a temporary file
        audio_file_path = "temp_audio.wav"
        with open(audio_file_path, "wb") as f:
            f.write(audio_bytes)

        # Convert the audio to text using the speech_to_text function
        transcript = speech_to_text(audio_file_path)
        if transcript:
            st.session_state.messages.append({"role": "user", "content": transcript})
            with st.chat_message("user"):
                st.write(transcript)
            os.remove(audio_file_path)

if st.session_state.messages[-1]["role"] != "assistant":
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            final_response = get_answer(st.session_state.messages)
            st.session_state.messages.append({"role": "assistant", "content": final_response})
            st.write(final_response)

            with st.spinner("Generating audio response..."):
                audio_file = text_to_speech(final_response)
                st.audio(audio_file)


2025-12-13 19:54:10.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.281 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.281 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.282 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.284 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.285 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-13 19:54:10.286 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [9]:
!pip install gradio
!pip install audio_recorder

ERROR: Could not find a version that satisfies the requirement audio_recorder (from versions: none)
ERROR: No matching distribution found for audio_recorder


In [17]:
from openai import OpenAI
from gtts import gTTS
import speech_recognition as sr
import gradio as gr
import os

# Initialize OpenAI client
client = OpenAI(
    api_key="d217acfc-3842-4f43-9eba-2067b6fd21a5",
    base_url="https://api.sambanova.ai/v1"
)

def speech_to_text(audio_data):
    """Convert speech audio to text using SpeechRecognition."""
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_data) as source:
        audio = recognizer.record(source)  # Read the entire audio file
        try:
            transcript = recognizer.recognize_google(audio)
            return transcript
        except sr.UnknownValueError:
            return "Sorry, I could not understand the audio."
        except sr.RequestError as e:
            return f"Could not request results from Google Speech Recognition service; {e}"

def text_to_speech(input_text):
    """Convert text to speech using gTTS."""
    tts = gTTS(text=input_text, lang='en')
    audio_file_path = "response.mp3"
    tts.save(audio_file_path)
    return audio_file_path

def get_answer(messages):
    """Get a response from OpenAI's GPT model."""
    try:
        response = client.chat.completions.create(model="Meta-Llama-3.1-8B-Instruct", messages=messages)
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

def process_chat_input(text):
    """Process text input and generate chat response."""
    messages = [{"role": "user", "content": text}]
    response = get_answer(messages)
    audio_file = text_to_speech(response)
    return response, audio_file

def process_audio_and_chat(audio):
    """Process audio input and generate chat response."""
    transcript = speech_to_text(audio)
    if transcript:
        return process_chat_input(transcript)

# Gradio interface
with gr.Blocks(css=".dark-theme { background-color: #2a2a2a; color: black; }") as demo:
    gr.Markdown("<h1 style='text-align: center;'>AskBot</h1>", elem_id="header")

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type="filepath")  # For audio input
            text_input = gr.Textbox(label="Type your message here:", placeholder="Type your message...")
            submit_btn = gr.Button("Submit")

    output_text = gr.Textbox(label="Response", interactive=False)
    output_audio = gr.Audio(label="Audio Response")

    def handle_submission(audio, text):
        if audio:
            response, audio_file = process_audio_and_chat(audio)
        elif text:
            response, audio_file = process_chat_input(text)
        else:
            return "Please provide an input."

        return response, audio_file  # Return values directly

    submit_btn.click(handle_submission, inputs=[audio_input, text_input], outputs=[output_text, output_audio])

demo.launch()


/tmp/ipython-input-3373332220.py:55: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=".dark-theme { background-color: #2a2a2a; color: black; }") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20adffd41fb29f215f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
